In [ ]:
# Path configuration
from pathlib import Path

PROJECT_ROOT = Path("..")  # Repository root.
BASELINE = PROJECT_ROOT / "data"
FIRM_DATA = PROJECT_ROOT
PATSTAT = PROJECT_ROOT / "patstat"
PATENT_TEXT = PROJECT_ROOT / "patent_text"
SAMPLEDATA = PROJECT_ROOT / "sampledata"
DATA = Path("data")          # Generated revision intermediates.
OUTPUTS = Path("outputs")    # Tables, figures and saved comparison results.
SEED = 42


# FSE embeddings and four-field validation scores

Prepare title/abstract vectors for `../0_get_familiy_id.ipynb` and `../1_ccmt_matching.ipynb`, and four-field candidate scores for SI S2.2. Run from `dataprep/` with paths set in the first cell. Each text field is encoded with FSE Average and PARANMT-300, then matched and scored at the family level.

Install `../revision_notebooks/requirements.txt`, `../revision_notebooks/requirements-optional.txt` and `requirements-fse.txt`. The FSE source build requires Git and a C compiler; PARANMT-300 is loaded through FSE's pretrained-vector interface.


In [ ]:
import re
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb


TEXT_FILE = PATENT_TEXT / 'g_patent_title_abstract.parquet'
FSE = PATENT_TEXT / 'fse_embedding'
WORK = DATA / 'upstream/fse'
FSE.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
BASELINE.mkdir(parents=True, exist_ok=True)

FIELDS = ('abstract', 'summary', 'claim', 'detail')
ARRAY_NAMES = {
    'abstract': 'title_abstract_embedding.npy',
    'summary': 'summary_embedding.npy',
    'claim': 'claim_embedding.npy',
    'detail': 'detail_embedding.npy',
}
ID_NAMES = {
    'abstract': 'title_abstract_patent_id.parquet',
    'summary': 'summary_patent_id.parquet',
    'claim': 'claim_patent_id.parquet',
    'detail': 'detail_patent_id.parquet',
}
FIELD_SOURCES = {
    'summary': ('summary/g_brf_sum_text_*.parquet', 'summary_text'),
    'claim': ('claim/g_claims_*.parquet', 'claim_text'),
    'detail': ('detail/g_detail_desc_text_*.parquet', 'description_text'),
}
TEXT_BATCH = 5_000
VECTOR_BATCH = 50_000
QUERY_BATCH = 10_000
SCORE_BATCH = 100_000
THREADS = 8
ID_SCHEMA = pa.schema([('patent_id', pa.string())])


## Tokenizer and averaging model

Concatenate title and abstract with one space. Remove punctuation, lowercase, split on whitespace, and average the word vectors for each document. Missing or empty text produces undefined vectors. Process documents in batches to limit memory use.


In [ ]:
def tokenize(text):
    text = re.sub(r'[^\w\s]', '', str(text))
    return text.lower().split()


def encode_batch(texts, vectors):
    from fse import Average, IndexedList

    texts = texts.reset_index(drop=True)
    tokens = texts.loc[texts.notna()].map(tokenize)
    tokens = tokens.loc[tokens.map(len) > 0]
    encoded = np.full((len(texts), vectors.vector_size), np.nan, dtype=np.float32)
    if len(tokens):
        model = Average(vectors)
        model.train(IndexedList(tokens.tolist()))
        encoded[tokens.index.to_numpy()] = model.sv.vectors
    return encoded


In [ ]:
from fse import Vectors

word_vectors = Vectors.from_pretrained('paranmt-300')


## Title/abstract vectors in publication order

Read `patent_id,patent_title,patent_abstract`. Save one vector per input row, preserving the order used by `../0_get_familiy_id.ipynb` and `../1_ccmt_matching.ipynb`. The companion ID file records that order.


In [ ]:
source = pq.ParquetFile(TEXT_FILE)
embeddings = np.lib.format.open_memmap(
    FSE / ARRAY_NAMES['abstract'], mode='w+', dtype=np.float32,
    shape=(source.metadata.num_rows, word_vectors.vector_size),
)
row = 0
with pq.ParquetWriter(FSE / ID_NAMES['abstract'], ID_SCHEMA) as writer:
    for batch in source.iter_batches(
        batch_size=TEXT_BATCH,
        columns=['patent_id', 'patent_title', 'patent_abstract'],
    ):
        frame = batch.to_pandas()
        text = frame.patent_title + ' ' + frame.patent_abstract
        end = row + len(frame)
        embeddings[row:end] = encode_batch(text, word_vectors)
        writer.write_table(pa.Table.from_pydict(
            {'patent_id': frame.patent_id.astype(str).tolist()}, schema=ID_SCHEMA,
        ))
        row = end
embeddings.flush()
print('Title/abstract vectors:', embeddings.shape)
del embeddings


## Summary, ordered claims and detailed description

Read the yearly text files and join claims in `claim_sequence` order with blank lines. Encode each complete field with the tokenizer and averaging model above. Each field has its own vector array and companion ID file, ordered by patent ID. DuckDB assembles and streams text batches for encoding.


In [ ]:
con = duckdb.connect()
for field, (pattern, text_column) in FIELD_SOURCES.items():
    con.read_parquet(str(PATENT_TEXT / pattern)).create_view('field_source', replace=True)
    if field == 'claim':
        query = """
            SELECT CAST(patent_id AS VARCHAR) AS patent_id,
                   string_agg(claim_text, '\n\n' ORDER BY claim_sequence) AS text
            FROM field_source GROUP BY patent_id ORDER BY patent_id
        """
    else:
        query = f"""
            SELECT CAST(patent_id AS VARCHAR) AS patent_id, {text_column} AS text
            FROM field_source ORDER BY patent_id
        """
    n_rows = con.sql(f'SELECT count(*) FROM ({query})').fetchone()[0]
    embeddings = np.lib.format.open_memmap(
        FSE / ARRAY_NAMES[field], mode='w+', dtype=np.float32,
        shape=(n_rows, word_vectors.vector_size),
    )
    row = 0
    with pq.ParquetWriter(FSE / ID_NAMES[field], ID_SCHEMA) as writer:
        for batch in con.execute(query).fetch_record_batch(TEXT_BATCH):
            frame = batch.to_pandas()
            end = row + len(frame)
            embeddings[row:end] = encode_batch(frame.text, word_vectors)
            writer.write_table(pa.Table.from_pydict(
                {'patent_id': frame.patent_id.tolist()}, schema=ID_SCHEMA,
            ))
            row = end
    embeddings.flush()
    print(field, embeddings.shape)
    del embeddings
con.close()


## Field-specific family representatives

Run `../0_get_familiy_id.ipynb` before this section. Its `BASELINE/patvecdf.parquet` supplies publication `patent_id` and `docdb_family_id`; publication IDs omit the `US` prefix. For each field, select the first mapped publication per family in that field's saved row order.

A Y02/Y04 assignment identifies a green family. Normalize family vectors for cosine retrieval and scoring. Missing text and zero-norm vectors have undefined cosine and are stored as NaN.


In [ ]:
patfam = pd.read_parquet(
    BASELINE / 'patvecdf.parquet', columns=['patent_id', 'docdb_family_id'],
).dropna().drop_duplicates('patent_id')
patfam['patent_id'] = patfam.patent_id.astype(str)
patfam['docdb_family_id'] = patfam.docdb_family_id.astype(np.int64)
cpc = pd.read_parquet(PATSTAT / 'tls225.parquet', columns=['docdb_family_id', 'cpc4d'])
green_families = cpc.loc[
    cpc.cpc4d.astype(str).str[:3].isin(['Y02', 'Y04']), 'docdb_family_id',
].unique()

for field in FIELDS:
    folder = WORK / field
    folder.mkdir(parents=True, exist_ok=True)
    ids = pd.read_parquet(FSE / ID_NAMES[field]).reset_index(names='embedding_row')
    families = (ids.merge(patfam, on='patent_id', how='left', sort=False)
                .dropna(subset=['docdb_family_id'])
                .drop_duplicates('docdb_family_id').reset_index(drop=True))
    families['docdb_family_id'] = families.docdb_family_id.astype(np.int64)
    families['ccmt'] = families.docdb_family_id.isin(green_families)
    raw = np.load(FSE / ARRAY_NAMES[field], mmap_mode='r')
    normalized = np.lib.format.open_memmap(
        folder / 'family_vectors.npy', mode='w+', dtype=np.float32,
        shape=(len(families), raw.shape[1]),
    )
    available = np.zeros(len(families), dtype=bool)
    for start in range(0, len(families), VECTOR_BATCH):
        end = start + VECTOR_BATCH
        values = raw[families.embedding_row.iloc[start:end].to_numpy()]
        norm = np.linalg.norm(values, axis=1, keepdims=True)
        unit = np.divide(values, norm, out=np.full_like(values, np.nan), where=norm > 0)
        normalized[start:end] = unit
        available[start:end] = np.isfinite(unit).all(axis=1)
    normalized.flush()
    families['available'] = available
    families.to_parquet(folder / 'family_index.parquet', index=False)
    print(field, len(families), 'families;', int(available.sum()), 'defined vectors')
    del raw, normalized


## Ten nongreen neighbours for each of the four fields

Build a cosine HNSW index with default construction settings and query ten nongreen neighbours per green family. Use finite vectors for both index construction and queries. Save field-specific matches under `DATA/upstream/fse/`; `cos` is cosine distance.


In [ ]:
import nmslib

MATCH_SCHEMA = pa.schema([
    ('ccmt_patent_id', pa.int64()), ('patent_id', pa.int64()),
    ('cos', pa.float64()), ('rank', pa.int64()),
])
for field in FIELDS:
    folder = WORK / field
    families = pd.read_parquet(folder / 'family_index.parquet')
    vectors = np.load(folder / 'family_vectors.npy', mmap_mode='r')
    focal_rows = np.flatnonzero(families.ccmt & families.available)
    other_rows = np.flatnonzero(~families.ccmt & families.available)
    focal_ids = families.docdb_family_id.to_numpy()[focal_rows]
    other_ids = families.docdb_family_id.to_numpy()[other_rows]
    index = nmslib.init(method='hnsw', space='cosinesimil')
    index.addDataPointBatch(vectors[other_rows])
    index.createIndex(print_progress=True)
    index.saveIndex(str(folder / 'hnsw_cos_index_patvec_other.bin'), save_data=True)
    with pq.ParquetWriter(folder / 'matched_ccmt_other.parquet', MATCH_SCHEMA) as writer:
        for start in range(0, len(focal_rows), QUERY_BATCH):
            matched = index.knnQueryBatch(
                vectors[focal_rows[start:start + QUERY_BATCH]], k=10, num_threads=THREADS,
            )
            records = []
            for focal, (neighbors, distances) in zip(focal_ids[start:start + QUERY_BATCH], matched):
                records.extend((int(focal), int(other_ids[j]), float(distance))
                               for j, distance in zip(neighbors, distances))
            matches = pd.DataFrame(records, columns=['ccmt_patent_id', 'patent_id', 'cos'])
            matches = matches.sort_values(['ccmt_patent_id', 'cos', 'patent_id'])
            matches['rank'] = matches.groupby('ccmt_patent_id').cumcount() + 1
            writer.write_table(pa.Table.from_pandas(matches, schema=MATCH_SCHEMA, preserve_index=False))
    del vectors, index


## Union the four candidate lists and score every pair in all fields

Each row identifies a pair of families. Take the union of pairs retrieved in the four fields, then calculate all four cosine similarities for each pair. Missing fields receive NaN. The composite score is the sum of the four similarities for pairs with all four scores.


In [ ]:
streams = []
for field in FIELDS:
    stream = pd.read_parquet(
        WORK / field / 'matched_ccmt_other.parquet', columns=['ccmt_patent_id', 'patent_id'],
    ).assign(field=field)
    streams.append(stream)
pairs = (pd.concat(streams, ignore_index=True)
         .groupby(['ccmt_patent_id', 'patent_id'], sort=True).field.nunique()
         .rename('n_fields_retrieved').reset_index()
         .rename(columns={'ccmt_patent_id': 'ccmt_docdb_family_id',
                          'patent_id': 'docdb_family_id'}))
del streams, stream
pairs.head()


In [ ]:
for field in FIELDS:
    families = pd.read_parquet(WORK / field / 'family_index.parquet')
    vectors = np.load(WORK / field / 'family_vectors.npy', mmap_mode='r')
    family_index = pd.Index(families.docdb_family_id)
    focal = family_index.get_indexer(pairs.ccmt_docdb_family_id)
    other = family_index.get_indexer(pairs.docdb_family_id)
    similarity = np.full(len(pairs), np.nan, dtype=np.float32)
    for start in range(0, len(pairs), SCORE_BATCH):
        stop = min(start + SCORE_BATCH, len(pairs))
        position = np.arange(start, stop)
        present = (focal[start:stop] >= 0) & (other[start:stop] >= 0)
        position = position[present]
        similarity[position] = np.einsum(
            'ij,ij->i', vectors[focal[position]], vectors[other[position]],
        )
    pairs[field] = similarity
    del vectors
pairs['score'] = pairs[list(FIELDS)].sum(axis=1, min_count=4).astype(np.float32)
pairs.head()


## Publication IDs and saved validation scores

Use the first title/abstract publication per family as the displayed patent. Save both publication and family IDs in `BASELINE/candidate_pair_scores.parquet`. Publication IDs omit the `US` prefix here; validation task preparation adds it. The composite `score` is the four-field sum.


In [ ]:
publications = patfam.drop_duplicates('docdb_family_id')[['patent_id', 'docdb_family_id']]
scores = (pairs.merge(publications, on='docdb_family_id')
          .merge(publications.rename(columns={
              'patent_id': 'ccmt_patent_id', 'docdb_family_id': 'ccmt_docdb_family_id',
          }), on='ccmt_docdb_family_id'))
scores = scores[[
    'ccmt_patent_id', 'patent_id', 'ccmt_docdb_family_id', 'docdb_family_id',
    'abstract', 'summary', 'claim', 'detail', 'score', 'n_fields_retrieved',
]]
scores.to_parquet(BASELINE / 'candidate_pair_scores.parquet', index=False)
scores.head()


Run `../1_ccmt_matching.ipynb` after the title/abstract vectors and family concordance are ready. Then run `../revision_notebooks/01_prepare_validation.ipynb` with `BASELINE/candidate_pair_scores.parquet` to prepare validation tasks.
